In [3]:
from google.cloud import storage

BUCKET_NAME = "cognito-gcs"
SOURCE_ROOT = "Cognito_new/reports_test_phase/"
DESTINATION_ROOT = "Cognito_new/reports/"
MEDICAL_POTENTIAL_FOLDER = "Medical Potential/"


def get_molecule_names(bucket):
    """Return molecule names found below the test-phase reports prefix."""
    molecule_names = set()
    for blob in bucket.list_blobs(prefix=SOURCE_ROOT):
        relative_path = blob.name[len(SOURCE_ROOT):]
        path_parts = relative_path.split("/")
        if (
            len(path_parts) >= 2
            and path_parts[0]
            and path_parts[1] == MEDICAL_POTENTIAL_FOLDER.rstrip("/")
        ):
            molecule_names.add(path_parts[0])
    return sorted(molecule_names)


def copy_molecule_reports(bucket, molecule_name, dry_run=True):
    """Copy PDF reports for one molecule, excluding its archive folder."""
    source_prefix = f"{SOURCE_ROOT}{molecule_name}/{MEDICAL_POTENTIAL_FOLDER}"
    destination_prefix = f"{DESTINATION_ROOT}{molecule_name}/{MEDICAL_POTENTIAL_FOLDER}"
    copied_count = 0

    source_blobs = list(bucket.list_blobs(prefix=source_prefix))
    for source_blob in source_blobs:
        relative_path = source_blob.name[len(source_prefix):].lstrip("/")
        if not relative_path.lower().endswith(".pdf"):
            continue
        if relative_path.lower().split("/", 1)[0] == "archive":
            continue

        destination_name = f"{destination_prefix}{relative_path}"
        print(f"{'Would copy' if dry_run else 'Copying'}: {source_blob.name} -> {destination_name}")
        copied_count += 1

        if not dry_run:
            bucket.copy_blob(source_blob, bucket, destination_name)

    return copied_count



# Set to False to perform the copy after reviewing the dry-run output.
DRY_RUN = False

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
molecule_names = get_molecule_names(bucket)
print(f"Found {len(molecule_names)} molecule(s): {molecule_names}")

total_reports = 0
for molecule_name in molecule_names:
    total_reports += copy_molecule_reports(bucket, molecule_name, dry_run=DRY_RUN)

action = "to copy" if DRY_RUN else "copied"
print(f"{total_reports} PDF report(s) {action}.")

Found 85 molecule(s): ['Acmopatide', 'Albenatide', 'Albiglutide', 'Aleniglipron L-Arginine', 'Asc-30', 'Azd-6234+Azd-9550', 'Azd-9550', 'Benaglutide', 'Berobenatide', 'Bgm-0504', 'Bi-3034701', 'Bofanglutide', 'Brenipatide', 'Buntanetap Tartrate+Dulaglutide', 'Buntanetap Tartrate+Dulaglutide+Sildenafil Citrate', 'Cagrilintide+Semaglutide', 'Cotadutide', 'Ct-996', 'Da-302168S', 'Dapiglutide', 'Dulaglutide', 'Dyx-116', 'Ecnoglutide', 'Efinopegdutide', 'Efocipegtrutide', 'Efpeglenatide', 'Elecoglipron', 'Enicepatide', 'Exenatide', 'Gsbr-1290', 'Hdm-1002', 'Hdm-1005', 'Hm-15275', 'Hrs-7535', 'Hs-10501', 'Hs-20004', 'Insulin Degludec+Liraglutide', 'Insulin Icodec+Semaglutide', 'Ixekizumab+Tirzepatide', 'Liraglutide', 'Liraglutide Biobetter', 'Lixisenatide', 'Lotiglipron', 'Mazdutide', 'Mdr-001', 'Met-224O', 'Mirikizumab+Tirzepatide', 'Mrank-001', 'Mwn-109', 'Na-931', 'Naperiglipron Erbumine', 'Nn-9490', 'Olatorepatide', 'Opk-88006', 'Orforglipron Calcium', 'Pb-718', 'Pegapamodutide', 'Peglox